# Employee Departure Prediction using Machine Learning

## Introduction

This project is aimed at building and evaluating various machine learning models for an employee departure classification problem. The dataset used in this analysis contains information about employees, such as their gender, distance from the office, years of experience, salary, and performance reviews, as well as a target variable indicating whether the employee left the company or not.

## Objective

- The primary objective of this analysis is to develop a reliable machine learning model.
- The model aims to accurately predict whether an employee is likely to leave the company.
- Predictions are based on given features or variables related to employees.
- This information is valuable for organizations as it helps identify and address potential issues leading to employee attrition.
- The ultimate goal is to assist organizations in retaining their valuable workforce by proactively managing employee retention strategies.

In [1]:
# Importing Necessary libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

In [2]:
import warnings
warnings.filterwarnings('ignore')

**1. Data Loading and Initial Inspection**

In [3]:

file_path = "/Users/sherinfeno/Desktop/DATA-602/mid term/employee_departure_dataset.csv"
df = pd.read_csv(file_path)

# Displaying data information and first rows
df.info()
df.head()
df.describe()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/sherinfeno/Desktop/DATA-602/mid term/employee_departure_dataset.csv'

**Explanation** :

This dataset has 300,000 employee records, covering demographics, work metrics, engagement, and performance.

Identifiers: RecordId (unique), Gender (binary).
Experience & Work: YearsWorked (0-14 years), TrainingHours, WorkLifeBalance, NumOfProjects, JobInvolvement, TeamSize, MentorshipReceived.
Engagement & Satisfaction: Scores for SkillDevelopmentCourses, ProjectComplexity, WorkSatisfaction, and JobEngagement.
Health & Wellbeing: PhysicalActivityScore and MentalWellbeingScore (1-9 scale).
Reviews: SelfReview(scored 3-5) and SupervisorReview (scored 2-5).
Department & Turnover: DepartmentCode (1-7) and Left (binary).


**2. Data Cleaning**

In [ ]:
# Data cleaning: replacing and converting fields
df['Distance'] = df['Distance'].replace({
    '>30miles': 30, '~10miles': 10, '<5mile': 5, '~15miles': 15, '~20miles': 20
}).astype(float)
df['PreviousSalary'] = pd.to_numeric(df['PreviousSalary'].str.replace('K', ''), errors='coerce') * 1000
df['Salary'] = pd.to_numeric(df['Salary'].str.replace('K', ''), errors='coerce') * 1000

# missing values imputation
df.fillna(df.median(), inplace=True)

# Dropping redundant columns
df = df.drop(columns=['Unnamed: 0'])
df = df.drop(columns=['RecordId'])


Explanation: 

Replacing and Converting String-Based Numeric Fields: The 'Distance', 'PreviousSalary', and 'Salary' fields contain string-based values (e.g., ">30miles", "59K") that are converted into appropriate numeric formats for analysis.
Handling Missing Values: We use the median to impute missing values, as it is a robust measure that is not heavily influenced by outliers.
Dropping Redundant Columns: The 'Unnamed: 0' column, which likely represents row indices, is removed to avoid any unintended influence on model predictions. The column RecordsID is also removed. 
By cleaning the data, we ensure it is in a usable format and minimize any errors or inconsistencies that may arise during analysis.



**3. Exploratory Data Analysis (EDA)**


3.1 Target Variable Analysis

In [ ]:
import seaborn as sns

# Distribution of the target variable
sns.countplot(x='Left', data=df)
plt.title('Distribution of Target Variable: Left')
plt.xlabel('Left (1 = Yes, 0 = No)')
plt.ylabel('Count')
plt.show()


Explanation
The target variable 'Left' represents whether an employee has left the organization (1 = Yes, 0 = No). Analyzing the distribution of the target variable helps us understand class balance:

Class Imbalance: If one class dominates, it may impact model performance, requiring adjustments (e.g., class weighting or resampling techniques).
General Trends: Insights on employee retention versus departure proportions can guide feature engineering and model choice.

3.2 Analysis of Numerical Variables

In [ ]:
# Plotting histograms for numerical columns
numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns.drop('Left')
df[numerical_columns].hist(bins=20, figsize=(15, 15))
plt.suptitle('Histograms of Numerical Features')
plt.show()


In [ ]:
# Correlation matrix
plt.figure(figsize=(15, 10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix of Features')
plt.show()


Here are some key insights:

Low to Moderate Correlations:

1 . The majority of the features show low or near-zero correlation with each other. This suggests that most features are not highly collinear, which can be beneficial for model training as it reduces redundancy and multicollinearity. Low correlation between features means each contributes unique information, allowing the model to consider multiple aspects of the data.
Key Features with Noticeable Correlation:

2 . A few features, like SelfReview and SupervisorReview, show a slight negative or positive correlation with each other, but these correlations are not strong enough to suggest multicollinearity issues. However, their relationship could indicate that employees with higher self-reviews tend to have a certain level of supervisor review scores as well.
PreviousSalary and Salary have some correlation, which is expected, as past salaries might influence current salaries within the organization.
Target Variable ("Left") Correlations:

3 . The "Left" column, representing whether an employee left the organization or not, has a low but noticeable correlation with features like SupervisorReview, DepartmentCode, and StressLevel. These correlations align with the feature importance findings, where these features were among the most influential in predicting employee departures.
SupervisorReview shows a negative correlation with "Left," indicating that employees with higher supervisor review scores are less likely to leave, potentially reflecting that positive evaluations by supervisors contribute to higher retention.
StressLevel has a positive correlation with "Left," suggesting that higher stress levels are associated with a higher likelihood of leaving, reinforcing the importance of addressing employee well-being for retention strategies.
Workplace Balance and Engagement Features:

4 . Features like WorkLifeBalance, JobEngagementScore, and WorkSatisfactionScore show low correlations with "Left." Although they don’t have strong correlations in this matrix, they may still contribute meaningfully when interacting with other features in the model. These aspects are often complex, and their impact might not be fully captured by a simple correlation.
Departmental and Role-Based Factors:

5 . DepartmentCode shows a slight correlation with "Left," implying that the department an employee belongs to might influence their likelihood of staying or leaving. This could be due to differences in work culture, resources, or management practices across departments.
TeamSize and NumOfProjects also exhibit minimal correlation with "Left," indicating that workload or team dynamics may not be direct indicators of employee turnover in this dataset.


**4. Feature Engineering**


Creating New Features

In [ ]:
# Creating new features
df['Engagement_Satisfaction_Ratio'] = df['JobEngagementScore'] / df['WorkSatisfactionScore']
df['Performance_Satisfaction'] = df['PeerFeedbackScore'] * df['WorkSatisfactionScore']
df['YearsWorked_Category'] = pd.cut(df['YearsWorked'], bins=[0, 3, 10, df['YearsWorked'].max()], labels=['New', 'Experienced', 'Senior'])
df['Salary_Range'] = pd.cut(df['Salary'], bins=[0, 55000, 65000, df['Salary'].max()], labels=['Low', 'Medium', 'High'])
df['Overall_Satisfaction'] = df['WorkSatisfactionScore'] + df['JobEngagementScore'] + df['WorkLifeBalance']
df['Health_Wellbeing_Score'] = df['PhysicalActivityScore'] + df['MentalWellbeingScore']
df['High_Job_Involvement'] = df['JobInvolvement'].apply(lambda x: 1 if x > 3 else 0)
df['Mentorship_Received'] = df['MentorshipReceived'].astype(bool)
df['Overworked'] = df.apply(lambda row: 1 if row['NumOfProjects'] > 10 and row['WorkLifeBalance'] < 3 else 0, axis=1)
df['Skill_Development_Activity'] = df['Certifications'] + df['OnsiteOpportunities'] + df['SkillDevelopmentCourses']
df['Growth_Engagement'] = df['SkillDevelopmentCourses'] * df['JobEngagementScore']


The rationale for creating each engineered feature and its relevance to predicting employee retention:

1. Engagement_Satisfaction_Ratio
Purpose: This ratio compares an employee’s engagement score relative to their satisfaction level at work.
Insight: Higher engagement relative to satisfaction may indicate employees who are actively contributing but potentially dissatisfied with certain aspects of their job, which could correlate with a higher likelihood of departure. This feature allows us to detect discrepancies between engagement and satisfaction.
2. Performance_Satisfaction
Purpose: Created by multiplying the peer feedback score with work satisfaction, this feature provides a combined measure of how performance and satisfaction align.
Insight: High values might suggest employees who are both satisfied and perform well, potentially indicating stability in their role. Low values, however, may signal employees who are either underperforming or dissatisfied, both of which are risk factors for leaving.
3. YearsWorked_Category
Purpose: This categorical feature divides employees into tenure groups (New, Experienced, and Senior).
Insight: Employees with shorter tenures may be more prone to turnover, whereas more tenured employees may have a greater sense of loyalty. Segmenting employees based on tenure allows the model to treat the likelihood of leaving differently based on experience level.
4. Salary_Range
Purpose: Categorizes salary into Low, Medium, and High ranges.
Insight: Employees in lower salary brackets may have higher turnover risks due to potential dissatisfaction with compensation. This feature allows for a more straightforward comparison of salary levels across employees.
5. Overall_Satisfaction
Purpose: Combines work satisfaction, engagement, and work-life balance into a single metric representing an employee’s general job satisfaction.
Insight: A high overall satisfaction score may indicate a lower risk of turnover, while low scores may highlight employees with multiple areas of dissatisfaction.
6. Health_Wellbeing_Score
Purpose: Aggregates physical activity and mental wellbeing scores.
Insight: Employee wellness can significantly impact retention. Those with poor health or wellbeing may be more likely to experience burnout, making this score valuable for assessing turnover risk related to employee health.
7. High_Job_Involvement
Purpose: This binary feature identifies employees with high involvement (scores above 3).
Insight: High involvement may signal a stronger connection to the job, potentially lowering turnover risk. Conversely, employees with low involvement may be less committed to their role.
8. Mentorship_Received
Purpose: Converts the mentorship received indicator into a Boolean feature.
Insight: Access to mentorship may enhance job satisfaction, learning, and engagement, which are often correlated with lower turnover. This feature captures the effect of mentorship on retention.
9. Overworked
Purpose: This binary feature flags employees managing over 10 projects with a work-life balance score below 3.
Insight: Overworked employees with poor work-life balance may face higher burnout and turnover risks. This feature is designed to identify employees under potentially unsustainable workloads.
10. Skill_Development_Activity
Purpose: Aggregates certifications, onsite opportunities, and skill development courses to create a cumulative skill development score.
Insight: Employees involved in skill development activities may feel valued and more likely to stay. This feature reflects an employee’s professional growth opportunities.
11. Growth_Engagement
Purpose: Captures the interaction between skill development and job engagement.
Insight: High engagement in skill development activities may reduce turnover by fostering a sense of growth and investment in the organization. This feature is intended to gauge the effect of growth opportunities on employee retention.
These engineered features were selected to capture different facets of job satisfaction, engagement, workload, tenure, and wellbeing—factors that can directly impact an employee's decision to stay or leave. By incorporating these nuanced features, the model can make more informed predictions regarding employee retention, considering both job-related and personal factors.








**5. Data Preprocessing and Splitting**

In [ ]:
# ensuring 'YearsWorked_Category' and 'Salary_Range' are created
df['YearsWorked_Category'] = pd.cut(df['YearsWorked'], bins=[0, 3, 10, df['YearsWorked'].max()], labels=['New', 'Experienced', 'Senior'])
df['Salary_Range'] = pd.cut(df['Salary'], bins=[0, 55000, 65000, df['Salary'].max()], labels=['Low', 'Medium', 'High'])

# One-hot encoding categorical features and scaling numerical features
df = pd.get_dummies(df, columns=['YearsWorked_Category', 'Salary_Range'], drop_first=True)

# Defining the list of numerical features for scaling
numerical_features = ['YearsWorked', 'TrainingHours', 'Distance', 'Salary', 'PreviousSalary', 
                      'Engagement_Satisfaction_Ratio', 'Performance_Satisfaction', 
                      'Overall_Satisfaction', 'Health_Wellbeing_Score']

# Initializing the scaler and apply it to numerical features
scaler = StandardScaler()
df[numerical_features] = scaler.fit_transform(df[numerical_features])

# Splitting data into features (X) and target (y)
X = df.drop(columns=['Left'])
y = df['Left']

# Splitting the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


Explanation: 

This preprocessing step standardizes numerical features and encodes categorical features, preparing data for model input. This ensures:

Scaled Features: Prevents model bias toward larger numeric ranges.


Categorical Encoding: Allows categorical variables to be represented in numerical form for compatibility with algorithms.

**6. Model Building ,Evaluation and Balancing the data with SMOTE (Synthetic Minority Over-sampling Technique)**


For predicting employee turnover, AUC-ROC, precision, recall, and F1-score are preferred over accuracy due to potential class imbalance. These metrics capture the model’s ability to detect leavers accurately, balancing false positives and negatives, and ensuring effective identification of at-risk employees for targeted retention strategies.


SMOTE (Synthetic Minority Over-sampling Technique) addresses class imbalance by generating synthetic samples for the minority class (employees who left). This helps the model learn more balanced patterns.

6.1 Logistic Regression Model (Baseline) with SMOTE (Handling Class Imbalance)



In [ ]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Initializing and fitting Logistic Regression on the resampled data
log_reg = LogisticRegression(random_state=42, max_iter=500)
log_reg.fit(X_train_resampled, y_train_resampled)

# Making predictions on the test set
y_pred_log_reg = log_reg.predict(X_test)
y_proba_log_reg = log_reg.predict_proba(X_test)[:, 1]

# Evaluation metrics
print("Classification Report:\n", classification_report(y_test, y_pred_log_reg))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log_reg))
print("AUC-ROC Score:", roc_auc_score(y_test, y_proba_log_reg))


The logistic regression model achieves 78% accuracy, with better performance on "stay" (0) predictions (precision: 0.84, recall: 0.85) than on "leave" (1) predictions (precision: 0.65, recall: 0.63). The AUC-ROC score of 0.84 indicates decent overall class separation. However, misclassifications (false positives/negatives) remain significant in predicting leavers.


## Decision Tree Model
A decision tree classifier is utilized to predict employee departures. Decision trees make predictions based on features like years of experience and performance reviews. 

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE

# Applying SMOTE to the training data only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Initializing and fitting the Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_resampled, y_train_resampled)

# Making predictions on the test set
y_pred_dt = dt_model.predict(X_test)
y_proba_dt = dt_model.predict_proba(X_test)[:, 1]  # Probabilities for AUC-ROC

# Evaluation metrics
print("Classification Report:\n", classification_report(y_test, y_pred_dt))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("AUC-ROC Score:", roc_auc_score(y_test, y_proba_dt))


Explanation: The untuned decision tree model achieves 84% accuracy, with higher precision and recall for "stay" (0) predictions (0.88) compared to "leave" (1) predictions (0.74). The AUC-ROC of 0.81 suggests fair class distinction but room for improvement, especially in reducing false positives and capturing leavers more effectively. 

**Hyperparameter tuning of Decision Tree Classifier**

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTE

# Defining the parameter grid
param_dist = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 6],
    'max_features': ['sqrt', 'log2', None],
    'criterion': ['gini', 'entropy']
}

# Initializing SMOTE and apply it to the training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Initializing the Decision Tree Classifier
dt = DecisionTreeClassifier(random_state=42)

# Setting up RandomizedSearchCV with cross-validation
random_search = RandomizedSearchCV(estimator=dt, param_distributions=param_dist, 
                                   n_iter=50, cv=5, scoring='roc_auc', 
                                   n_jobs=-1, verbose=2, random_state=42)

# Fitting RandomizedSearchCV
random_search.fit(X_train_resampled, y_train_resampled)

# Displaying the best parameters and AUC-ROC score
print("Best Parameters:", random_search.best_params_)
print("Best AUC-ROC Score from RandomizedSearchCV:", random_search.best_score_)

In [ ]:
# Using the best model from RandomizedSearchCV
best_dt_model = random_search.best_estimator_

# Prediction on the test set
y_pred_dt_best = best_dt_model.predict(X_test)
y_proba_dt_best = best_dt_model.predict_proba(X_test)[:, 1]

# Printing classification report, confusion matrix, and AUC-ROC score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Classification Report:\n", classification_report(y_test, y_pred_dt_best))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_dt_best))
print("AUC-ROC Score:", roc_auc_score(y_test, y_proba_dt_best))


The tuned decision tree achieves 85% accuracy, with improved recall for "leave" (1) predictions (0.86) and precision for "stay" (0) predictions (0.93). An AUC-ROC of 0.92 shows strong class separation, reducing false negatives and enhancing leaver detection, making it more effective for retention-focused predictions.

# 6.2 Random Forest Model with SMOTE (Handling Class Imbalance)

In [ ]:


# Handlling class imbalance with SMOTE
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

# Training the Random Forest model on resampled data
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_res, y_train_res)

# Evaluation on the test set
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]
print("Random Forest Report:\n", classification_report(y_test, y_pred_rf))
print("Random Forest AUC-ROC Score:", roc_auc_score(y_test, y_proba_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))


Model Performance: Using classification metrics and AUC-ROC score, we evaluate the Random Forest model’s performance on the test set, providing insights into how well it differentiates between employees who stay and those who leave.

The untuned random forest achieves 80% accuracy, with high recall for "stay" (0) predictions (0.89) but lower performance for "leave" (1) predictions (recall: 0.61). The AUC-ROC of 0.89 suggests good class separation, though high false negatives indicate room for improvement in identifying potential leavers.

7. Hyperparameter Tuning with RandomizedSearchCV

In [ ]:
# Defining parameter grid for RandomizedSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2']
}
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_grid, 
                                   n_iter=50, cv=3, scoring='roc_auc', n_jobs=-1, random_state=42)
random_search.fit(X_train_res, y_train_res)

# Extracting the best model and evaluate
best_rf = random_search.best_estimator_
y_pred_best_rf = best_rf.predict(X_test)
y_proba_best_rf = best_rf.predict_proba(X_test)[:, 1]

print("Tuned Random Forest Report:\n", classification_report(y_test, y_pred_best_rf))
print("Tuned Random Forest AUC-ROC Score:", roc_auc_score(y_test, y_proba_best_rf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_best_rf))



Explanation:

The tuned Random Forest model shows strong performance, with an accuracy of 79% and an AUC-ROC of 0.89, indicating good discrimination. Class 0 predictions are notably better than Class 1, achieving higher precision, recall, and F1-score. The confusion matrix reveals a challenge with false negatives in Class 1 predictions.

# **Feature Importance and Interpretation**


By visualizing the top 15 features, we identify key factors contributing to employee departure.
Model Transparency: Understanding feature importance enhances transparency and can guide strategic actions, such as focusing on employee engagement or satisfaction factors shown to impact departure likelihood.

**8. Retraining with Top 15 Features Only has improved accuracy**


In [ ]:
# Training the Random Forest mode
best_rf.fit(X_train, y_train)  

# Creating the feature importances DataFrame
feature_importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

# Selecting the top 20 features by importance
top_features = feature_importances.iloc[:15]['Feature'].values

# Filtering the training and testing sets to include only these features
X_train_refined = X_train[top_features]
X_test_refined = X_test[top_features]

# Displaying the selected features
print("Selected Top Features for Model:")
print(top_features)

# Visualizing the top features to reconfirm
plt.figure(figsize=(10, 6))
plt.barh(feature_importances.iloc[:15]['Feature'], feature_importances.iloc[:15]['Importance'])
plt.xlabel("Importance")
plt.title("Top 15 Feature Importances")
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Re-training the model with the refined features
best_rf.fit(X_train_refined, y_train)

# Making predictions and evaluate
y_pred_refined = best_rf.predict(X_test_refined)
y_proba_refined = best_rf.predict_proba(X_test_refined)[:, 1]

# Classification report and evaluation metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("Classification Report (Top Features):\n", classification_report(y_test, y_pred_refined))
print("Confusion Matrix (Top Features):\n", confusion_matrix(y_test, y_pred_refined))
print("AUC-ROC Score (Top Features):", roc_auc_score(y_test, y_proba_refined))


The tuned random forest model achieves 84% accuracy, with precision of 0.90 and recall of 0.86 for "stay" (0) predictions, and 0.72 precision and 0.78 recall for "leave" (1) predictions. An AUC-ROC score of 0.91 indicates strong class separation and improved identification of potential leavers.

**Conclusions and Recommendations**



This analysis examines various machine learning models for classifying employee departures, with a focus on the tuned decision tree and random forest models. The tuned decision tree model outperforms others in accuracy and recall for identifying potential leavers, achieving an AUC-ROC score of 0.92. While it excels in predictive performance, factors like interpretability and computational efficiency remain critical for real-world application. The decision tree model's simplicity aids in communicating findings to stakeholders, while the random forest model, though slightly less effective in this case, may be preferable in scenarios requiring scalability with larger datasets. Ultimately, the tuned decision tree is the recommended choice for effectively identifying at-risk employees while ensuring clarity in interpretation.
